# 04 — Generation Demo

Load a trained checkpoint and generate text from custom prompts.
Compare outputs across different temperatures and top-k values.

In [1]:
import sys
sys.path.insert(0, '../..')

import torch
import tiktoken

from src.infra.config import load_config
from src.infra.io import load_checkpoint
from src.models.gpt.model import GPT
from src.core.generation import generate

In [5]:
# ── Load config and checkpoint ────────────────────────────────────────────
config = load_config('../../configs/experiments/exp_001_baseline.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPT(config.model).to(device)
ckpt_path = '../../outputs/checkpoints/best_model.pt'
load_checkpoint(ckpt_path, model, device=device)
model.eval()

enc = tiktoken.get_encoding('gpt2')
print(f'Model loaded on {device}')

Model loaded on cpu


In [6]:
# ── Generate text ─────────────────────────────────────────────────────────
def generate_text(prompt: str, max_tokens: int = 200,
                  temperature: float = 1.0, top_k: int = 50) -> str:
    tokens = enc.encode_ordinary(prompt)
    ctx = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    out = generate(model, ctx, max_new_tokens=max_tokens,
                   temperature=temperature, top_k=top_k)
    return enc.decode(out.squeeze().tolist())

prompts = [
    'Once upon a time there was a pumpkin.',
    'A little girl went to the woods',
]
for p in prompts:
    print(f'--- Prompt: "{p}" ---')
    print(generate_text(p, max_tokens=200))
    print()

--- Prompt: "Once upon a time there was a pumpkin." ---
Once upon a time there was a pumpkin. It had soft soft eyes. It was the most beautiful pumpkin in the forest!"

The fox was so amazed and excited. The turkey asked his mom, "What is this coin?"

"No, it's not good," he said.

His mom laughed too. "But it's too dangerous. We don't want to touch it!"

The bunny didn't want to give up. It was a good idea! He was curious and decided to look for the mushroom. So, he thought for a small way to get the coin for him.

Suddenly, something unusual happened. A boy turned around and saw the bee. The fairy said, "Where are you going?" The duck said, "I was looking for treasure! Have you like it?"

The deer turned down and said, "Okay, I was so beautiful!" They looked at each other and felt very sad. The wolf said, "I wish I had never tasted it yet.

--- Prompt: "A little girl went to the woods" ---
A little girl went to the woods to show her mom. She said, "Mom, that's not a real frog. This fr

In [7]:
# ── Temperature sweep ─────────────────────────────────────────────────────
prompt = 'Once upon a time'
for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f'--- temperature={temp} ---')
    print(generate_text(prompt, max_tokens=80, temperature=temp, top_k=50))
    print()

--- temperature=0.5 ---
Once upon a time, there was a little girl named Lily. She loved to play outside in the park. One day, she saw a bird flying in the sky. It was so pretty! She picked it up and started to fly.

When she was done, she saw a big bird hopping around in the sky. She was so excited and asked her mom if she could go. Her mom said yes and

--- temperature=0.8 ---
Once upon a time, there was a boy named Timmy. Timmy loved to play in the park. One day, Timmy's mommy came to visit him. She was scared because she didn't know what to do. Suddenly, Timmy realized that she could have anything to do. 

"I don't like Lily, we need to put a new toy for you," replied his mom.

--- temperature=1.0 ---
Once upon a time, there was a little puppy named Jack. Jack was three year old and had a big red fur. Jack loved to play with his toys, but he wasn't sure what to do. But one day, his owner told him to go to the park and find a very special friend. Jack was so excited by his adventure 